In [1]:
import numpy as np
import pandas as pd
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import SolverFactory

In [59]:
lista = [i for i in range(1,10)]

dicio = {i:i for i in lista}

In [64]:
model = pyo.ConcreteModel()

model.set_p = pyo.Set(initialize=lista)
model.s = pyo.Var(model.set_p, within=NonNegativeReals)
model.y = pyo.Var(model.set_p,domain=pyo.Binary)
model.p = pyo.Var(model.set_p, within=NonNegativeReals)
model.profit = pyo.Param(model.set_p,initialize=dicio)
#Restrições
# Se y3 = 1 entao y1 e y2 também precisam ser 1
def restricao1(model):
    return model.y[3] <= model.y[1]
model.restricao1 = pyo.Constraint(rule=restricao1)
def restricao2(model):
    return model.y[3] <= model.y[2]
model.restricao2 = pyo.Constraint(rule=restricao2)

def restricao3(model):
    return model.y[6] + model.y[7] + model.y[8] <= 2
model.restricao3 = pyo.Constraint(rule=restricao3)

def restricao4(model):
    return model.y[3] + model.y[4] + 2*model.y[5] <= 2
model.restricao4 = pyo.Constraint(rule=restricao4)

# objetivo

def objetivo(model):
    return sum(model.profit[i]*model.y[i] for i in model.set_p)
model.objetivo = pyo.Objective(rule=objetivo, sense=maximize)


In [63]:
model.pprint()

1 Set Declarations
    set_p : Size=1, Index=None, Ordered=Insertion
        Key  : Dimen : Domain : Size : Members
        None :     1 :    Any :    9 : {1, 2, 3, 4, 5, 6, 7, 8, 9}

1 Param Declarations
    profit : Size=9, Index=set_p, Domain=Any, Default=None, Mutable=False
        Key : Value
          1 :     1
          2 :     2
          3 :     3
          4 :     4
          5 :     5
          6 :     6
          7 :     7
          8 :     8
          9 :     9

3 Var Declarations
    p : Size=9, Index=set_p
        Key : Lower : Value : Upper : Fixed : Stale : Domain
          1 :     0 :  None :  None : False :  True : NonNegativeReals
          2 :     0 :  None :  None : False :  True : NonNegativeReals
          3 :     0 :  None :  None : False :  True : NonNegativeReals
          4 :     0 :  None :  None : False :  True : NonNegativeReals
          5 :     0 :  None :  None : False :  True : NonNegativeReals
          6 :     0 :  None :  None : False :  True : Non

In [65]:
# ------------------- solver
opt = SolverFactory('cplex', executable='C:\\Program Files\\IBM\\ILOG\\CPLEX_Studio_Community222\\cplex\\bin\\x64_win64\\cplex.exe')
res = opt.solve(model,tee=True)


Welcome to IBM(R) ILOG(R) CPLEX(R) Interactive Optimizer Community Edition 22.2.0.0
  with Simplex, Mixed Integer & Barrier Optimizers
5725-A06 5725-A29 5724-Y48 5724-Y49 5724-Y54 5724-Y55 5655-Y21
Copyright IBM Corp. 1988, 2026.  All Rights Reserved.

Type 'help' for a list of available commands.
Type 'help' followed by a command name for more
information on commands.

CPLEX> Logfile 'cplex.log' closed.
Logfile 'C:\Users\Pichau\AppData\Local\Temp\tmp0wt9znx0.cplex.log' open.
CPLEX> Problem 'C:\Users\Pichau\AppData\Local\Temp\tmpyek5r44h.pyomo.lp' read.
Read time = 0.00 sec. (0.00 ticks)
CPLEX> Problem name         : C:\Users\Pichau\AppData\Local\Temp\tmpyek5r44h.pyomo.lp
Objective sense      : Maximize
Variables            :       9  [Binary: 9]
Objective nonzeros   :       9
Linear constraints   :       4  [Less: 4]
  Nonzeros           :      10
  RHS nonzeros       :       2

Variables            : Min LB: 0.000000         Max UB: 1.000000       
Objective nonzeros   : Min   : 1.0

In [66]:
for a in model.set_p:
    print(f'Ativo {a}: {model.y[a].value:.4f}')

Ativo 1: 1.0000
Ativo 2: 1.0000
Ativo 3: 1.0000
Ativo 4: 1.0000
Ativo 5: 0.0000
Ativo 6: 0.0000
Ativo 7: 1.0000
Ativo 8: 1.0000
Ativo 9: 1.0000
